In [ ]:
import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from transformers import AutoTokenizer
from transformers.utils import logging
from wordcloud import WordCloud
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    roc_auc_score
)

1

In [ ]:
train = pd.read_csv("train.csv")

print("Исходный размер:", train.shape)

print("\nКоличество пропусков:")
print(train.isna().sum())

duplicates_count = train.duplicated(
    subset=["Prompt", "Answer", "Target"]
).sum()

print("\nКоличество дубликатов:", duplicates_count)

train = train.drop_duplicates(
    subset=["Prompt", "Answer", "Target"],
    keep="first"
)

train = train.dropna(
    subset=["Prompt", "Answer", "Target"]
)

print("\nРазмер после очистки:", train.shape)

2

In [ ]:
print(train["Target"].value_counts())
print(train["Target"].value_counts(normalize=True))

train["Target"].value_counts().sort_index().plot(kind="bar")

plt.xlabel("Target")
plt.ylabel("Количество")
plt.title("Распределение классов")
plt.show()

class_counts = train["Target"].value_counts()
class_shares = train["Target"].value_counts(normalize=True)
minority_class = class_counts.idxmin()
majority_class = class_counts.idxmax()

print(
    f"Интерпретация: класс {minority_class} составляет "
    f"{class_shares[minority_class]:.2%} выборки, а класс {majority_class} — "
    f"{class_shares[majority_class]:.2%}. Выборка заметно несбалансирована, "
    "поэтому одной Accuracy недостаточно для оценки качества модели."
)

In [ ]:
train["Prompt_len"] = train["Prompt"].str.len()
train["Answer_len"] = train["Answer"].str.len()
plt.figure(figsize=(12, 6))
plt.hist(train["Prompt_len"], bins=50, alpha=0.7, label="Prompt")
plt.hist(train["Answer_len"], bins=50, alpha=0.7, label="Answer")

plt.xlabel("Длина текста в символах")
plt.ylabel("Количество текстов")
plt.title("Распределение длин Prompt и Answer")
plt.legend()

plt.show()

print(
    f"Интерпретация: медианная длина Prompt — {train['Prompt_len'].median():.0f} символов, "
    f"95-й перцентиль — {train['Prompt_len'].quantile(0.95):.0f}, "
    f"максимум — {train['Prompt_len'].max():.0f}."
)
print(
    f"Для Answer медиана — {train['Answer_len'].median():.0f}, "
    f"95-й перцентиль — {train['Answer_len'].quantile(0.95):.0f}, "
    f"максимум — {train['Answer_len'].max():.0f}. "
    "Правый хвост распределения показывает наличие очень длинных текстов, "
    "которые нужно просмотреть отдельно."
)

In [ ]:
pd.set_option("display.max_colwidth", None)


# !!! В А Ж Н О !!!
''' я убрал ограничение на вывод текста,
чтобы просмотреть примеры длинных и коротких ответов модели '''
''' надо использовать clear cell output
для того чтобы не было проблемы с прокруткой страницы с моим кодом'''

In [ ]:
train.loc[train["Id"] == 19947, "Target"] = 1
train.loc[train["Id"] == 22155, "Target"] = 0
train.loc[train["Id"] == 4132, "Target"] = 0

print(train.nsmallest(10, "Answer_len")[
    ["Id", "Prompt", "Answer", "Answer_len", "Target"]
])
#здесь я указываю id а не индексы
#19947 (a: 5) модель просят вывести букву ответа (a), а модель вывела содержание буквы ответа (5)
#значит то-что target = 0 это неправильно, заменил на target = 1
#22155 модель выводит букву ответа как её и ожидают, буква (c) есть в вариантах ответа
#значит то-что target = 1 это неправильно, заменил на target = 0
#4132 модель просят вывести описательное предложение, а модель выводит точку
#значит то-что target = 1 это неправильно, заменил на target = 0
#1046 модель не обьяснила ответ, хотя ответ 0 это один из возможных ответов даже если неправильный
#оставил как есть, сложный кейс, модель всё таки ничего не выдумала



In [ ]:
for _, row in train.nlargest(10, "Answer_len").iterrows():
    print("Id:", row["Id"])
    print("Prompt:", row["Prompt"])
    print("Answer:", row["Answer"])
    print("Answer_len:", row["Answer_len"])
    print("Target:", row["Target"])
    print("-" * 100)

In [ ]:
#25512 ответ выходит за рамки опций
train.loc[train["Id"] == 25512, "Target"] = 1
#16857 ответ не удовлетворяет запросу
train.loc[train["Id"] == 16857, "Target"] = 1


3

In [ ]:
logging.set_verbosity_error()
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

train["Prompt_tokens"] = train["Prompt"].apply(tokenizer.tokenize)
train["Answer_tokens"] = train["Answer"].apply(tokenizer.tokenize)

all_tokens = pd.concat([
    train["Prompt_tokens"].explode(),
    train["Answer_tokens"].explode()
])

token_counts = all_tokens.value_counts()

top_20 = token_counts.head(20)

print(top_20)

plt.figure(figsize=(12, 6))

plt.bar(
    top_20.index,
    top_20.values
)

plt.xlabel("Токен")
plt.ylabel("Частота")
plt.title("Топ-20 самых частых токенов")
plt.xticks(rotation=45)

plt.show()

wordcloud = WordCloud(
    width=1200,
    height=600,
    background_color="black"
).generate_from_frequencies(top_20.to_dict())

plt.figure(figsize=(15, 7))

plt.imshow(
    wordcloud,
    interpolation="bilinear"
)

plt.axis("off")
plt.title("Облако топ-20 самых частых токенов")

plt.show()

print(
    "Интерпретация: среди самых частых токенов много служебных и шаблонных слов. "
    "Они встречаются часто, но обычно несут меньше смысловой информации, поэтому "
    "на следующем шаге часть таких токенов удаляется."
)

In [ ]:
stop_words = {
    "the", "a", "to", "and", "of", "is", "in", "that", "are", "##t",
    "ins", "as"
}

def clean_tokens(tokens):
    return [
        token
        for token in tokens
        if token not in stop_words
        and token.isalpha()
    ]

train["Prompt_tokens_clean"] = train["Prompt_tokens"].apply(clean_tokens)
train["Answer_tokens_clean"] = train["Answer_tokens"].apply(clean_tokens)

all_clean_tokens = pd.concat([
    train["Prompt_tokens_clean"].explode(),
    train["Answer_tokens_clean"].explode()
])

clean_token_counts = all_clean_tokens.value_counts()

top_20_clean = clean_token_counts.head(20)

print(top_20_clean)

plt.figure(figsize=(12, 6))

plt.bar(
    top_20_clean.index,
    top_20_clean.values
)

plt.xlabel("Токен")
plt.ylabel("Частота")
plt.title("Топ-20 токенов после очистки")
plt.xticks(rotation=45)

plt.show()

wordcloud = WordCloud(
    width=1200,
    height=600,
    background_color="black"
).generate_from_frequencies(top_20_clean.to_dict())

plt.figure(figsize=(15, 7))

plt.imshow(
    wordcloud,
    interpolation="bilinear"
)

plt.axis("off")
plt.title("Облако слов после очистки")

plt.show()

removed_share = 1 - len(all_clean_tokens) / len(all_tokens)

print(
    f"Интерпретация: после удаления небольшого ручного набора стоп-слов "
    f"и небуквенных токенов удалено около {removed_share:.2%} токенов. "
    "В частотном списке стало меньше технического шума и больше содержательных слов."
)

4

In [ ]:
x = train["Prompt"] + " " + train["Answer"]
y = train["Target"]

x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.2,
    random_state=34,
    stratify=y
)

vectorizer = TfidfVectorizer()

x_train = vectorizer.fit_transform(x_train)
x_test = vectorizer.transform(x_test)



In [ ]:
model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced"
)

start_time = time.perf_counter()
model.fit(x_train, y_train)
baseline_train_time = time.perf_counter() - start_time

start_time = time.perf_counter()
y_pred = model.predict(x_test)
y_score = model.predict_proba(x_test)[:, 1]
baseline_inference_time = time.perf_counter() - start_time

baseline_accuracy = accuracy_score(y_test, y_pred)
baseline_precision = precision_score(y_test, y_pred, pos_label=1, zero_division=0)
baseline_recall = recall_score(y_test, y_pred, pos_label=1, zero_division=0)
baseline_f1 = f1_score(y_test, y_pred, pos_label=1, zero_division=0)
baseline_balanced_accuracy = balanced_accuracy_score(y_test, y_pred)
baseline_roc_auc = roc_auc_score(y_test, y_score)

print("Accuracy:", baseline_accuracy)
print("Precision класса 1:", baseline_precision)
print("Recall класса 1:", baseline_recall)
print("F1 класса 1:", baseline_f1)
print("Balanced Accuracy:", baseline_balanced_accuracy)
print("ROC-AUC:", baseline_roc_auc)
print("Время обучения:", baseline_train_time)
print("Время инференса:", baseline_inference_time)

print(
    "Интерпретация: из-за дисбаланса классов Accuracy рассматривается вместе "
    "с Precision, Recall и F1 редкого класса 1, а также Balanced Accuracy и ROC-AUC."
)

In [ ]:
feature_names = vectorizer.get_feature_names_out()

weights = model.coef_[0]

importance = pd.DataFrame({
    "token": feature_names,
    "weight": weights
})

top_positive = importance.nlargest(20, "weight")

print("Топ-20 токенов, влияющих на Target = 1:")
print(top_positive)

plt.figure(figsize=(12, 6))
plt.bar(
    top_positive["token"],
    top_positive["weight"]
)
plt.xticks(rotation=45)
plt.xlabel("Токен")
plt.ylabel("Коэффициент")
plt.title("Топ-20 токенов, влияющих на Target = 1")
plt.show()

top_negative = importance.nsmallest(20, "weight")

print("Топ-20 токенов, влияющих на Target = 0:")
print(top_negative)

plt.figure(figsize=(12, 6))
plt.bar(
    top_negative["token"],
    top_negative["weight"]
)
plt.xticks(rotation=45)
plt.xlabel("Токен")
plt.ylabel("Коэффициент")
plt.title("Топ-20 токенов, влияющих на Target = 0")
plt.gca().invert_yaxis()
plt.show()

print(
    "Интерпретация: положительные коэффициенты сильнее сдвигают предсказание "
    "в сторону Target = 1, отрицательные — в сторону Target = 0. "
    "Чем больше абсолютное значение коэффициента, тем сильнее влияние токена."
)

5

In [ ]:
x = (
    train["Prompt_tokens_clean"].apply(lambda tokens: " ".join(tokens))
    + " "
    + train["Answer_tokens_clean"].apply(lambda tokens: " ".join(tokens))
)

y = train["Target"]

x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.2,
    random_state=34,
    stratify=y
)

vectorizer = TfidfVectorizer()

x_train_vec = vectorizer.fit_transform(x_train)
x_test_vec = vectorizer.transform(x_test)

model_clean = LogisticRegression(
    max_iter=1000,
    class_weight="balanced"
)

model_clean.fit(x_train_vec, y_train)

y_pred_clean = model_clean.predict(x_test_vec)
y_score_clean = model_clean.predict_proba(x_test_vec)[:, 1]

clean_accuracy = accuracy_score(y_test, y_pred_clean)
clean_precision = precision_score(y_test, y_pred_clean, pos_label=1, zero_division=0)
clean_recall = recall_score(y_test, y_pred_clean, pos_label=1, zero_division=0)
clean_f1 = f1_score(y_test, y_pred_clean, pos_label=1, zero_division=0)
clean_balanced_accuracy = balanced_accuracy_score(y_test, y_pred_clean)
clean_roc_auc = roc_auc_score(y_test, y_score_clean)

comparison_clean = pd.DataFrame(
    {
        "Исходный бейзлайн": [
            baseline_accuracy,
            baseline_precision,
            baseline_recall,
            baseline_f1,
            baseline_balanced_accuracy,
            baseline_roc_auc
        ],
        "Очищенные тексты": [
            clean_accuracy,
            clean_precision,
            clean_recall,
            clean_f1,
            clean_balanced_accuracy,
            clean_roc_auc
        ]
    },
    index=[
        "Accuracy",
        "Precision класса 1",
        "Recall класса 1",
        "F1 класса 1",
        "Balanced Accuracy",
        "ROC-AUC"
    ]
)

comparison_clean["Разница"] = (
    comparison_clean["Очищенные тексты"]
    - comparison_clean["Исходный бейзлайн"]
)

print("Сравнение очищенных текстов с исходным бейзлайном шага 4:")
print(comparison_clean)

if clean_f1 > baseline_f1:
    print(
        "Интерпретация: после очистки F1 класса 1 улучшился относительно "
        "исходного бейзлайна."
    )
elif clean_f1 < baseline_f1:
    print(
        "Интерпретация: после очистки F1 класса 1 ухудшился относительно "
        "исходного бейзлайна. Значит, часть удалённых токенов содержала "
        "полезный для классификации сигнал."
    )
else:
    print(
        "Интерпретация: очистка не изменила F1 класса 1 относительно "
        "исходного бейзлайна."
    )

feature_names = vectorizer.get_feature_names_out()
weights = model_clean.coef_[0]

importance = pd.DataFrame({
    "token": feature_names,
    "weight": weights
})

top_positive = importance.nlargest(
    20,
    "weight"
)
print("\nТоп-20 токенов, влияющих на Target = 1:")
print(top_positive)

plt.figure(figsize=(12, 6))

plt.bar(
    top_positive["token"],
    top_positive["weight"]
)

plt.xticks(rotation=45)
plt.xlabel("Токен")
plt.ylabel("Коэффициент")
plt.title("Топ-20 токенов, влияющих на Target = 1")
plt.show()

top_negative = importance.nsmallest(
    20,
    "weight"
)
print("\nТоп-20 токенов, влияющих на Target = 0:")
print(top_negative)

plt.figure(figsize=(12, 6))
plt.bar(
    top_negative["token"],
    top_negative["weight"]
)
plt.xticks(rotation=45)
plt.xlabel("Токен")
plt.ylabel("Коэффициент")
plt.title("Топ-20 токенов, влияющих на Target = 0")
plt.gca().invert_yaxis()
plt.show()

print(
    "Интерпретация: после очистки наиболее значимые положительные и отрицательные "
    "коэффициенты показывают, какие оставшиеся токены сильнее всего влияют "
    "на решение логистической регрессии."
)

importance["abs_weight"] = importance["weight"].abs()

n = 5000

top_tokens = importance.nlargest(
    n,
    "abs_weight"
)["token"]

top_tokens = set(top_tokens)

vectorizer_top = TfidfVectorizer(
    vocabulary=top_tokens
)

x_train_top = vectorizer_top.fit_transform(x_train)
x_test_top = vectorizer_top.transform(x_test)

model_top = LogisticRegression(
    max_iter=1000,
    class_weight="balanced"
)
model_top.fit(
    x_train_top,
    y_train
)

y_pred_top = model_top.predict(x_test_top)

top_f1 = f1_score(y_test, y_pred_top, pos_label=1, zero_division=0)

print("\nПосле отбора top 5000 токенов:")
print("F1 на очищенном тексте:", clean_f1)
print("F1 после отбора top 5000:", top_f1)
print("Разница с очищенным текстом:", top_f1 - clean_f1)
print("Разница с исходным бейзлайном:", top_f1 - baseline_f1)

6

In [ ]:
# если бы я не знал что pytorch популярнее tensorflow на рынке, то я бы не мучился сейчас, а использовал готовую фигню ;(
texts = (train["Prompt"] + " " + train["Answer"]).tolist()
labels = train["Target"].to_numpy()

encoded = tokenizer(
    texts,
    padding=True,
    truncation=True,
    max_length=256,
    return_tensors="pt"
)

x = encoded["input_ids"]
y = torch.tensor(labels, dtype=torch.float32)

x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.2,
    random_state=34,
    stratify=labels
)

class TextCNN(nn.Module):
        def __init__(self, vocab_size):
            super().__init__()
            self.embedding = nn.Embedding(
            vocab_size,
            128
        )
            self.conv = nn.Conv1d(
            in_channels=128,
            out_channels=128,
            kernel_size=3
        )
            self.pool = nn.AdaptiveMaxPool1d(1)
            self.fc = nn.Linear(
                128,
                1
        )
            self.sigmoid = nn.Sigmoid()
        def forward(self, x):
            x = self.embedding(x)
            x = x.permute(0, 2, 1)
            x = self.conv(x)
            x = torch.relu(x)
            x = self.pool(x)
            x = x.squeeze(2)
            x = self.fc(x)
            return self.sigmoid(x).squeeze(1)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)
model = TextCNN(
    vocab_size=tokenizer.vocab_size
).to(device)
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

epochs = 5

train_loader = DataLoader(
    TensorDataset(x_train, y_train),
    batch_size=32,
    shuffle=True
)

test_loader = DataLoader(
    TensorDataset(x_test, y_test),
    batch_size=32
)

if device.type == "cuda":
    torch.cuda.synchronize()

start_time = time.perf_counter()

for epoch in range(epochs):
    model.train()
    total_loss = 0
    for x_batch, y_batch in train_loader:
        x_batch = x_batch.to(device)
        y_batch = y_batch.to(device)
        optimizer.zero_grad()
        predictions = model(x_batch)
        loss = criterion(
            predictions,
            y_batch
        )
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch + 1}:","Loss =", total_loss / len(train_loader))

if device.type == "cuda":
    torch.cuda.synchronize()

cnn_train_time = time.perf_counter() - start_time
print("Время обучения CNN:", cnn_train_time)

In [ ]:
model.eval()
predictions = []
prediction_scores = []
true_labels = []

if device.type == "cuda":
    torch.cuda.synchronize()

start_time = time.perf_counter()

with torch.no_grad():

    for x_batch, y_batch in test_loader:

        x_batch = x_batch.to(device)

        probabilities = model(x_batch)

        predicted_classes = (
            probabilities >= 0.5
        ).int()

        predictions.extend(
            predicted_classes.cpu().numpy()
        )

        prediction_scores.extend(
            probabilities.cpu().numpy()
        )

        true_labels.extend(
            y_batch.numpy()
        )

if device.type == "cuda":
    torch.cuda.synchronize()

cnn_inference_time = time.perf_counter() - start_time

cnn_accuracy = accuracy_score(true_labels, predictions)
cnn_precision = precision_score(
    true_labels,
    predictions,
    pos_label=1,
    zero_division=0
)
cnn_recall = recall_score(
    true_labels,
    predictions,
    pos_label=1,
    zero_division=0
)
cnn_f1 = f1_score(
    true_labels,
    predictions,
    pos_label=1,
    zero_division=0
)
cnn_balanced_accuracy = balanced_accuracy_score(
    true_labels,
    predictions
)
cnn_roc_auc = roc_auc_score(
    true_labels,
    prediction_scores
)

print("Accuracy:", cnn_accuracy)
print("Precision класса 1:", cnn_precision)
print("Recall класса 1:", cnn_recall)
print("F1 класса 1:", cnn_f1)
print("Balanced Accuracy:", cnn_balanced_accuracy)
print("ROC-AUC:", cnn_roc_auc)
print("Время обучения:", cnn_train_time)
print("Время инференса:", cnn_inference_time)

if cnn_accuracy > 0.9 and cnn_f1 < 0.3:
    print(
        "Интерпретация: высокая Accuracy при низком F1 класса 1 означает, "
        "что CNN в основном успешно распознаёт частый класс 0, но хуже работает "
        "с редким классом 1. Поэтому делать вывод о превосходстве CNN только "
        "по Accuracy нельзя."
    )
else:
    print(
        "Интерпретация: качество CNN нужно оценивать по метрикам редкого класса, "
        "Balanced Accuracy и ROC-AUC вместе с Accuracy."
    )

comparison = pd.DataFrame(
    {
        "TF-IDF + Logistic Regression": [
            baseline_accuracy,
            baseline_precision,
            baseline_recall,
            baseline_f1,
            baseline_balanced_accuracy,
            baseline_roc_auc,
            baseline_train_time,
            baseline_inference_time
        ],
        "CNN": [
            cnn_accuracy,
            cnn_precision,
            cnn_recall,
            cnn_f1,
            cnn_balanced_accuracy,
            cnn_roc_auc,
            cnn_train_time,
            cnn_inference_time
        ]
    },
    index=[
        "Accuracy",
        "Precision класса 1",
        "Recall класса 1",
        "F1 класса 1",
        "Balanced Accuracy",
        "ROC-AUC",
        "Train time, s",
        "Inference time, s"
    ]
)

print("\nСводное сравнение качества и скорости:")
print(comparison)